In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Helper to produce min/max timestamp strings in UTC+8
def ts_range_info(series):
    if series.empty:
        return {"min_timestamp": None, "max_timestamp": None}
    mn = int(series.min())
    mx = int(series.max())
    mn_s = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    mx_s = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    return {"min_timestamp": mn, "min_timestamp_readable": mn_s,
            "max_timestamp": mx, "max_timestamp_readable": mx_s}

# Summarize metric.csv
metric_total = len(metric_df)
metric_time = ts_range_info(metric_df['timestamp']) if 'timestamp' in metric_df.columns else {}
metric_cmdb_counts = metric_df['cmdb_id'].value_counts() if 'cmdb_id' in metric_df.columns else pd.Series(dtype=int)
metric_unique_cmdb_count = int(metric_cmdb_counts.shape[0])
metric_unique_cmdb_top = metric_cmdb_counts.index.tolist()[:50]

if 'kpi_name' in metric_df.columns:
    metric_top_kpis = (
        metric_df['kpi_name']
        .value_counts()
        .head(20)
        .reset_index()
        .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
    )
    metric_top_kpis_list = metric_top_kpis.to_dict(orient='records')
else:
    metric_top_kpis_list = []

metric_summary = {
    "total_rows": int(metric_total),
    **metric_time,
    "unique_cmdb_count": metric_unique_cmdb_count,
    "unique_cmdb_top_50": metric_unique_cmdb_top,
    "top_20_kpi_name_by_count": metric_top_kpis_list
}

# Summarize trace.csv
trace_total = len(trace_df)
trace_time = ts_range_info(trace_df['timestamp']) if 'timestamp' in trace_df.columns else {}
trace_cmdb_counts = trace_df['cmdb_id'].value_counts() if 'cmdb_id' in trace_df.columns else pd.Series(dtype=int)
trace_unique_cmdb_count = int(trace_cmdb_counts.shape[0])
trace_unique_cmdb_top = trace_cmdb_counts.index.tolist()[:50]

if 'trace_name' in trace_df.columns:
    trace_top_names = (
        trace_df['trace_name']
        .value_counts()
        .head(20)
        .reset_index()
        .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
    )
    trace_top_names_list = trace_top_names.to_dict(orient='records')
else:
    trace_top_names_list = []

trace_summary = {
    "total_rows": int(trace_total),
    **trace_time,
    "unique_cmdb_count": trace_unique_cmdb_count,
    "unique_cmdb_top_50": trace_unique_cmdb_top,
    "top_20_trace_name_by_count": trace_top_names_list
}

# Summarize log.csv
log_total = len(log_df)
log_time = ts_range_info(log_df['timestamp']) if 'timestamp' in log_df.columns else {}
log_cmdb_counts = log_df['cmdb_id'].value_counts() if 'cmdb_id' in log_df.columns else pd.Series(dtype=int)
log_unique_cmdb_count = int(log_cmdb_counts.shape[0])
log_unique_cmdb_top = log_cmdb_counts.index.tolist()[:50]

if 'log_name' in log_df.columns:
    log_top_names = (
        log_df['log_name']
        .value_counts()
        .head(20)
        .reset_index()
        .rename(columns={'index': 'log_name', 'log_name': 'count'})
    )
    log_top_names_list = log_top_names.to_dict(orient='records')
else:
    log_top_names_list = []

log_summary = {
    "total_rows": int(log_total),
    **log_time,
    "unique_cmdb_count": log_unique_cmdb_count,
    "unique_cmdb_top_50": log_unique_cmdb_top,
    "top_20_log_name_by_count": log_top_names_list
}

# Summarize error_logs.csv
error_total = len(error_df)
error_time = ts_range_info(error_df['timestamp']) if 'timestamp' in error_df.columns else {}
error_cmdb_counts = error_df['cmdb_id'].value_counts() if 'cmdb_id' in error_df.columns else pd.Series(dtype=int)
error_unique_cmdb_count = int(error_cmdb_counts.shape[0])
error_unique_cmdb_top = error_cmdb_counts.index.tolist()[:50]

# Up to 10 distinct messages (or first 10 rows). For each message, show first occurrence timestamp and cmdb_id.
if 'message' in error_df.columns and not error_df.empty:
    # Ensure message is string
    error_df['message'] = error_df['message'].astype(str)
    distinct_msgs = error_df['message'].dropna().unique().tolist()[:10]
    msgs_info = []
    for msg in distinct_msgs:
        subset = error_df[error_df['message'] == msg]
        first_row = subset.sort_values('timestamp').iloc[0]
        ts = int(first_row['timestamp'])
        ts_readable = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
        msgs_info.append({
            "message": msg,
            "first_timestamp": ts,
            "first_timestamp_readable": ts_readable,
            "cmdb_id": first_row.get('cmdb_id', None)
        })
else:
    # Fallback: first 10 rows
    msgs_info = []
    for _, row in error_df.head(10).iterrows():
        ts = int(row['timestamp']) if 'timestamp' in row and pd.notnull(row['timestamp']) else None
        ts_readable = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z') if ts is not None else None
        msgs_info.append({
            "message": str(row.get('message', None)),
            "first_timestamp": ts,
            "first_timestamp_readable": ts_readable,
            "cmdb_id": row.get('cmdb_id', None)
        })

error_summary = {
    "total_rows": int(error_total),
    **error_time,
    "unique_cmdb_count": error_unique_cmdb_count,
    "unique_cmdb_top_50": error_unique_cmdb_top,
    "up_to_10_distinct_messages_or_first_10_rows": msgs_info
}

# Final JSON-serializable summary
summary = {
    "metric.csv": metric_summary,
    "trace.csv": trace_summary,
    "log.csv": log_summary,
    "error_logs.csv": error_summary
}

summary

```
Out[1]:
```
summary = (
    "Summary of telemetry files (all times shown in UTC+8): "
    "metric.csv: 147,132 rows from 2022-03-21 05:00 to 2022-03-21 05:29; 59 unique cmdb_id "
    "(top examples: adservice, adservice2, frontend2-0, frontend-0, frontend-1, frontend-2, checkoutservice-0, productcatalogservice-2, cartservice-1, shippingservice-0); "
    "top KPI counts are grouped around 1,050 and 990 rows for many KPIs. "
    "trace.csv: 23,952 rows covering the same 05:00–05:29 window; 39 unique cmdb_id (top examples: frontend-1, frontend-2, checkoutservice-2, productcatalogservice-0, recommendationservice-0); "
    "top trace feature counts cluster around 584, 581, 418, 277, and 261. "
    "log.csv: 1,604 rows in the same window; 31 unique cmdb_id (examples: adservice-0/1/2, cartservice-0/1/2, currencyservice-0/1/2, frontend-2, frontend-1); "
    "top log_name counts show two dominant entries with 802 rows each. "
    "error_logs.csv: 453 rows from 2022-03-21 05:00:04 to 2022-03-21 05:29:58; only 2 unique cmdb_id (frontend-1 and frontend-2). "
    "Up to 10 distinct messages: the dataset includes at least the message 'severity: warning, message: failed to retrieve ads' first observed at 2022-03-21 05:00:04 on frontend-1. "
    "Overall note: metric and trace data are large and distributed across many services/nodes; error logs are concentrated on frontend-1/2 with a recurring ad-retrieval warning."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'total_rows': 147132, 'min_timestamp': 1647810000, 'min_timestamp_readable': '2022-03-21 05:00:00+0800', 'max_timestamp': 1647811740, 'max_timestamp_readable': '2022-03-21 05:29:00+0800', 'unique_cmdb_count': 59, 'unique_cmdb_top_50': ['adservice', 'adservice2', 'frontend2-0', 'frontend-0', 'frontend-1', 'frontend-2', 'checkoutservice-0', 'checkoutservice-2', 'checkoutservice-1', 'checkoutservice2-0', 'productcatalogservice-2', 'productcatalogservice-0', 'productcatalogservice-1', 'adservice2-0', 'cartservice-1', 'productcatalogservice2-0', 'cartservice-0', 'cartservice2-0', 'cartservice-2', 'adservice-0', 'adservice-2', 'adservice-1', 'shippingservice-0', 'recommendationservice-0', 'recommendationservice-1', 'recommendationservice-2', 'recommendationservice2-0', 'shippingservice-2', 'shippingservice-1', 'shippingservice2-0', 'currencyservice-1', 'currencyservice-0', 'currencyservice-2', 'currencyservice2-0', 'emailservice2-0', 'emailservice-2', 'emailservice-1', 'emailservice-0', 'paymentservice2-0', 'paymentservice-2', 'paymentservice-0', 'paymentservice-1', 'node-2', 'node-4', 'node-6', 'node-1', 'node-3', 'node-5', 'redis-cart2-0', 'redis-cart-0'], 'top_20_kpi_name_by_count': [{'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 1050}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}, {'count': 990}]}, 'trace.csv': {'total_rows': 23952, 'min_timestamp': 1647810000, 'min_timestamp_readable': '2022-03-21 05:00:00+0800', 'max_timestamp': 1647811740, 'max_timestamp_readable': '2022-03-21 05:29:00+0800', 'unique_cmdb_count': 39, 'unique_cmdb_top_50': ['frontend-1', 'frontend-2', 'checkoutservice-2', 'checkoutservice-1', 'checkoutservice-0', 'checkoutservice2-0', 'frontend2-0', 'productcatalogservice-0', 'productcatalogservice-1', 'productcatalogservice-2', 'recommendationservice-0', 'recommendationservice-2', 'recommendationservice-1', 'recommendationservice2-0', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'currencyservice-2', 'currencyservice-0', 'currencyservice-1', 'shippingservice-2', 'shippingservice-1', 'shippingservice-0', 'cartservice2-0', 'adservice-0', 'adservice-1', 'adservice-2', 'productcatalogservice2-0', 'currencyservice2-0', 'shippingservice2-0', 'paymentservice-0', 'emailservice-2', 'adservice2-0', 'paymentservice2-0', 'emailservice2-0', 'paymentservice-1', 'emailservice-1', 'paymentservice-2', 'emailservice-0'], 'top_20_trace_name_by_count': [{'count': 584}, {'count': 584}, {'count': 584}, {'count': 584}, {'count': 581}, {'count': 581}, {'count': 581}, {'count': 581}, {'count': 418}, {'count': 418}, {'count': 418}, {'count': 418}, {'count': 277}, {'count': 277}, {'count': 277}, {'count': 277}, {'count': 261}, {'count': 261}, {'count': 261}, {'count': 261}]}, 'log.csv': {'total_rows': 1604, 'min_timestamp': 1647810000, 'min_timestamp_readable': '2022-03-21 05:00:00+0800', 'max_timestamp': 1647811740, 'max_timestamp_readable': '2022-03-21 05:29:00+0800', 'unique_cmdb_count': 31, 'unique_cmdb_top_50': ['adservice-0', 'adservice-1', 'adservice-2', 'cartservice-0', 'cartservice-1', 'cartservice-2', 'currencyservice-0', 'frontend-2', 'currencyservice-1', 'currencyservice-2', 'frontend-1', 'recommendationservice-0', 'productcatalogservice-2', 'productcatalogservice-1', 'productcatalogservice-0', 'shippingservice-2', 'recommendationservice-2', 'shippingservice-0', 'shippingservice-1', 'recommendationservice-1', 'checkoutservice-2', 'paymentservice-0', 'checkoutservice-1', 'paymentservice-2', 'emailservice-2', 'paymentservice-1', 'emailservice-1', 'emailservice-0', 'checkoutservice-0', 'redis-cart-0', 'frontend-0'], 'top_20_log_name_by_count': [{'count': 802}, {'count': 802}]}, 'error_logs.csv': {'total_rows': 453, 'min_timestamp': 1647810004, 'min_timestamp_readable': '2022-03-21 05:00:04+0800', 'max_timestamp': 1647811798, 'max_timestamp_readable': '2022-03-21 05:29:58+0800', 'unique_cmdb_count': 2, 'unique_cmdb_top_50': ['frontend-1', 'frontend-2'], 'up_to_10_distinct_messages_or_first_10_rows': [{'message': 'severity: warning, message: failed to retrieve ads', 'first_timestamp': 1647810004, 'first_timestamp_readable': '2022-03-21 05:00:04+0800', 'cmdb_id': 'frontend-1'}]}}```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse metric_df if present in the kernel state, otherwise load it
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Validate required columns
required_cols = {'kpi_name', 'cmdb_id'}
if not required_cols.issubset(set(metric_df.columns)):
    raise ValueError(f"metric.csv must contain columns: {required_cols}")

# 1) Top 50 unique kpi_name values with counts
top_kpis_df = (
    metric_df['kpi_name']
    .value_counts()
    .head(50)
    .rename_axis('kpi_name')
    .reset_index(name='count')
)

# 2) All unique cmdb_id containing 'frontend' and 'adservice' with row counts
cmdb_series = metric_df['cmdb_id'].astype(str)

frontend_mask = cmdb_series.str.contains('frontend', na=False)
adservice_mask = cmdb_series.str.contains('adservice', na=False)

frontend_counts_df = (
    cmdb_series[frontend_mask]
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

adservice_counts_df = (
    cmdb_series[adservice_mask]
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# 3) For each cmdb_id in (2), top 20 kpi_name for that cmdb_id with counts
def top_kpis_for_cmdb(cmdb_id, top_n=20):
    s = (
        metric_df.loc[metric_df['cmdb_id'].astype(str) == cmdb_id, 'kpi_name']
        .value_counts()
        .head(top_n)
        .rename_axis('kpi_name')
        .reset_index(name='count')
    )
    return s.to_dict(orient='records')

frontend_kpis_by_cmdb = {
    row['cmdb_id']: top_kpis_for_cmdb(row['cmdb_id'], top_n=20)
    for _, row in frontend_counts_df.iterrows()
}

adservice_kpis_by_cmdb = {
    row['cmdb_id']: top_kpis_for_cmdb(row['cmdb_id'], top_n=20)
    for _, row in adservice_counts_df.iterrows()
}

# Assemble compact, JSON-serializable result
result = {
    "top_kpis": top_kpis_df.to_dict(orient='records'),                     # up to 50
    "frontend_cmdbs": frontend_counts_df.to_dict(orient='records'),        # all frontend cmdbs with counts
    "adservice_cmdbs": adservice_counts_df.to_dict(orient='records'),      # all adservice cmdbs with counts
    "frontend_kpis_by_cmdb": frontend_kpis_by_cmdb,                        # per-frontend cmdb top 20 kpis
    "adservice_kpis_by_cmdb": adservice_kpis_by_cmdb                       # per-adservice cmdb top 20 kpis
}

result

```
Out[2]:
```
Here’s a concise plain-English summary of the metric.csv analysis:

- Top KPIs (overall, top entries):
  - The highest-count KPI entries are mostly container metrics for node-5 (e.g., container.node-5.container_network_receive_packets_dropped.eth0, container.node-5.container_network_transmit_MB.eth0, etc.) with counts around 1,050.
  - A large group of other container/node KPIs (many node-5 container metrics) appear with counts ≈990.
  - (The full top-50 list was produced; top examples are the container.node-5 metrics noted above.)

- Frontend cmdb_ids and row counts:
  - frontend2-0: 5,280 rows
  - frontend-0: 4,920 rows
  - frontend-1: 4,680 rows
  - frontend-2: 4,500 rows
  - frontend (service-level): 120 rows

- Adservice cmdb_ids and row counts:
  - adservice: 10,230 rows
  - adservice2: 9,990 rows
  - adservice2-0: 3,060 rows
  - adservice-0: 2,580 rows
  - adservice-2: 2,580 rows
  - adservice-1: 2,580 rows

- Top KPIs per frontend cmdb_id (top 20 each, counts are small/consistent):
  - frontend2-0: mostly container.node-6.* metrics (cpu CFS periods/throttling, cpu usage, fs reads/writes, file descriptors), each count = 30.
  - frontend-0, frontend-1, frontend-2: mostly container.node-5.* metrics (cpu CFS periods/throttling, cpu usage, fs reads/writes, file descriptors), each count = 30.
  - frontend (service): app.http metrics (app.http.count, app.http.mrt, app.http.rr, app.http.sr), each count = 30.

- Top KPIs per adservice cmdb_id (top 20 each):
  - adservice: JVM/runtime and app grpc/http metrics (runtime.jvm_threads_state.WAITING, app.grpc.*, app.http.*, various runtime Java metrics), counts = 30 each.
  - adservice2: similar JVM/runtime metrics and some jvm_memory_pool entries, counts = 30 each.
  - adservice2-0: container.node-6.* metrics (cpu, fs, file descriptors), counts = 30 each.
  - adservice-0, adservice-1, adservice-2: container.node-5.* metrics (cpu, fs, file descriptors), counts = 30 each.

Takeaway: frontend and adservice families are well represented. Frontend pods mainly report container metrics tied to specific nodes (node-5 or node-6) while the adservice entries include both JVM/runtime and container/node metrics depending on the specific cmdb_id. Counts for per-cmdb top KPIs are uniform (mostly 30), indicating a regular sampling cadence.

The original code execution output of IPython Kernel is also provided below for reference:

{'top_kpis': [{'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.stopped', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_usage_MB./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_write_seconds./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_ulimits_soft.max_open_files', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_quota', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_shares', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_swap_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.iowaiting', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.running', 'count': 990}, {'kpi_name': 'container.node-5.container_threads', 'count': 990}, {'kpi_name': 'container.node-5.container_tasks_state.uninterruptible', 'count': 990}, {'kpi_name': 'container.node-5.container_threads_max', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_memory_reservation_limit_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_start_time_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgfault', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failcnt', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_cache', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_period', 'count': 990}, {'kpi_name': 'container.node-5.container_sockets', 'count': 990}, {'kpi_name': 'container.node-5.container_last_seen', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_mapped_file', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_rss', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_max_usage_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_swap', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_working_set_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_sector_writes./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_usage_MB', 'count': 990}], 'frontend_cmdbs': [{'cmdb_id': 'frontend2-0', 'count': 5280}, {'cmdb_id': 'frontend-0', 'count': 4920}, {'cmdb_id': 'frontend-1', 'count': 4680}, {'cmdb_id': 'frontend-2', 'count': 4500}, {'cmdb_id': 'frontend', 'count': 120}], 'adservice_cmdbs': [{'cmdb_id': 'adservice', 'count': 10230}, {'cmdb_id': 'adservice2', 'count': 9990}, {'cmdb_id': 'adservice2-0', 'count': 3060}, {'cmdb_id': 'adservice-0', 'count': 2580}, {'cmdb_id': 'adservice-2', 'count': 2580}, {'cmdb_id': 'adservice-1', 'count': 2580}], 'frontend_kpis_by_cmdb': {'frontend2-0': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_sector_reads./dev/vda1', 'count': 30}], 'frontend-0': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}], 'frontend-1': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}], 'frontend-2': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}], 'frontend': [{'kpi_name': 'app.http.count', 'count': 30}, {'kpi_name': 'app.http.mrt', 'count': 30}, {'kpi_name': 'app.http.rr', 'count': 30}, {'kpi_name': 'app.http.sr', 'count': 30}]}, 'adservice_kpis_by_cmdb': {'adservice': [{'kpi_name': 'runtime.jvm_threads_state.WAITING', 'count': 30}, {'kpi_name': 'app.grpc.count', 'count': 30}, {'kpi_name': 'app.grpc.mrt', 'count': 30}, {'kpi_name': 'app.grpc.rr', 'count': 30}, {'kpi_name': 'app.grpc.sr', 'count': 30}, {'kpi_name': 'app.http.count', 'count': 30}, {'kpi_name': 'app.http.mrt', 'count': 30}, {'kpi_name': 'app.http.rr', 'count': 30}, {'kpi_name': 'app.http.sr', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_LoadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_Verbose', 'count': 30}, {'kpi_name': 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'count': 30}, {'kpi_name': 'runtime.java_lang_Compilation_TotalCompilationTime', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'count': 30}], 'adservice2': [{'kpi_name': 'runtime.jvm_threads_state.WAITING', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_LoadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_TotalLoadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_UnloadedClassCount', 'count': 30}, {'kpi_name': 'runtime.java_lang_ClassLoading_Verbose', 'count': 30}, {'kpi_name': 'runtime.java_lang_Compilation_CompilationTimeMonitoringSupported', 'count': 30}, {'kpi_name': 'runtime.java_lang_Compilation_TotalCompilationTime', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.MarkSweepCompact', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.MarkSweepCompact', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.Copy', 'count': 30}, {'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_GcThreadCount.MarkSweepCompact', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_init.Eden_Space', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_init.Metaspace', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_init.Survivor_Space', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_init.Tenured_Gen', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_max.Code_Cache', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_max.Compressed_Class_Space', 'count': 30}, {'kpi_name': 'runtime.jvm_memory_pool_MB_max.Eden_Space', 'count': 30}], 'adservice2-0': [{'kpi_name': 'container.node-6.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-6.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-6.container_fs_sector_reads./dev/vda1', 'count': 30}], 'adservice-0': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}], 'adservice-2': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}], 'adservice-1': [{'kpi_name': 'container.node-5.container_cpu_cfs_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 30}, {'kpi_name': 'container.node-5.container_file_descriptors', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_inodes_free./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_io_time_weighted_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_limit_MB./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_read_seconds./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 30}, {'kpi_name': 'container.node-5.container_fs_sector_reads./dev/vda1', 'count': 30}]}}```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if available, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure required columns
assert {'timestamp', 'cmdb_id', 'kpi_name', 'value'}.issubset(metric_df.columns), "metric.csv missing required columns"

# Candidate cmdb_ids
candidates = [
    'frontend-1','frontend-2','frontend-0','frontend2-0',
    'adservice','adservice2','adservice-0','adservice-1','adservice-2',
    'node-5','node-6'
]

# Patterns to match (substrings)
patterns = [
    'network','transmit_errors','receive_packets_dropped','packets',
    'container_cpu','cpu','cfs_throttled','app.http.sr','app.grpc.sr',
    'app.http.count','app.grpc.count','memory','fs_reads','fs_writes'
]

# Incident window (inclusive)
window_start = 1647810000
window_end = 1647811799

# timezone for readable timestamps
tz = pytz.timezone('Asia/Shanghai')
def to_readable(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# Prepare a mapping of candidate -> matched kpi_names (limit 50)
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)

candidate_kpis = {}
for cm in candidates:
    # filter rows for this cmdb_id
    sub = metric_df[metric_df['cmdb_id'] == cm]
    if sub.empty:
        candidate_kpis[cm] = []
        continue
    # find kpi_names matching any pattern
    mask = pd.Series(False, index=sub.index)
    for p in patterns:
        mask = mask | sub['kpi_name'].str.contains(p, na=False)
    kpis = sub.loc[mask, 'kpi_name'].unique().tolist()[:50]
    candidate_kpis[cm] = kpis

# Now for each component-KPI compute global p95/p5 and counts, then window anomalies
rows = []
for cm, kpis in candidate_kpis.items():
    if not kpis:
        continue
    for k in kpis:
        series = metric_df[(metric_df['cmdb_id'] == cm) & (metric_df['kpi_name'] == k)]['value'].dropna().astype(float)
        full_count = int(series.shape[0])
        if full_count == 0:
            # skip if no data
            continue
        # compute global P95 and P5 using entire series BEFORE window filtering
        global_p95 = float(np.nanquantile(series.values, 0.95))
        global_p5 = float(np.nanquantile(series.values, 0.05))
        # points in incident window
        window_df = metric_df[
            (metric_df['cmdb_id'] == cm) &
            (metric_df['kpi_name'] == k) &
            (metric_df['timestamp'] >= window_start) &
            (metric_df['timestamp'] <= window_end)
        ].copy()
        points_in_window = int(window_df.shape[0])
        anomaly_count = 0
        earliest_anom_ts = None
        earliest_anom_val = None
        sample_anom_values = []  # list of dicts {timestamp, timestamp_readable, value}
        if points_in_window > 0:
            # determine tail type: if kpi contains '.sr' treat as lower-tail (<= P5); else upper-tail (>= P95)
            lower_tail = ('.sr' in k)
            if lower_tail:
                anoms = window_df[window_df['value'].astype(float) <= global_p5].copy()
            else:
                anoms = window_df[window_df['value'].astype(float) >= global_p95].copy()
            anomaly_count = int(anoms.shape[0])
            if anomaly_count > 0:
                # earliest anomaly by timestamp
                anoms_sorted_ts = anoms.sort_values('timestamp')
                earliest_row = anoms_sorted_ts.iloc[0]
                earliest_anom_ts = int(earliest_row['timestamp'])
                earliest_anom_val = float(earliest_row['value'])
                # sample anomaly values: up to 5 highest (upper-tail) or lowest (lower-tail)
                if lower_tail:
                    sampled = anoms.sort_values('value').head(5)
                else:
                    sampled = anoms.sort_values('value', ascending=False).head(5)
                # build sample list
                sample_anom_values = [
                    {
                        "timestamp": int(r['timestamp']),
                        "timestamp_readable": to_readable(int(r['timestamp'])),
                        "value": float(r['value'])
                    }
                    for _, r in sampled.iterrows()
                ]
        rows.append({
            "cmdb_id": cm,
            "kpi_name": k,
            "global_p95": global_p95,
            "global_p5": global_p5,
            "full_count": full_count,
            "points_in_window": points_in_window,
            "anomaly_count_in_window": anomaly_count,
            "earliest_anomaly_timestamp": int(earliest_anom_ts) if earliest_anom_ts is not None else None,
            "earliest_anomaly_value": float(earliest_anom_val) if earliest_anom_val is not None else None,
            "sample_anomaly_values": sample_anom_values
        })

# Assemble DataFrame and select top 20 by anomaly_count_in_window desc
result_df = pd.DataFrame(rows)
if result_df.empty:
    result_df = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","global_p95","global_p5","full_count",
        "points_in_window","anomaly_count_in_window",
        "earliest_anomaly_timestamp","earliest_anomaly_value","sample_anomaly_values"
    ])
else:
    result_df = result_df.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False])

# Limit to top 20 rows
summary_table = result_df.head(20).reset_index(drop=True)

# For compactness, convert earliest_anomaly_timestamp to readable as additional short field
def readable_ts_or_none(ts):
    return to_readable(ts) if ts is not None else None

if not summary_table.empty:
    summary_table['earliest_anomaly_timestamp_readable'] = summary_table['earliest_anomaly_timestamp'].apply(lambda x: readable_ts_or_none(x) if pd.notnull(x) else None)

# Final JSON-compatible result as list of dicts (limited to top 20)
result = summary_table.to_dict(orient='records')

result

```
Out[3]:
```
Summary (plain English):

- Key finding: frontend-1 shows widespread "anomalies" across many container/node-5 KPIs during the incident window (1647810000 → 1647811799 / 2022-03-21 05:00–05:29 UTC+8).
- Scope: For each reported KPI below, the full series has 30 points, points_in_window = 30, and anomaly_count_in_window = 30 (i.e., every sample in the window was flagged).
- Earliest anomaly for all reported KPIs: 2022-03-21 05:00:00+0800.

Top affected KPIs for frontend-1 (examples and typical values):
- CPU-related: container.node-5.container_cpu_cfs_throttled_periods (0.0), container.node-5.container_cpu_cfs_throttled_seconds (0.0), container.node-5.container_cpu_load_average_10s (0.0)
- Filesystem reads/writes: container.node-5.container_fs_reads./dev/vda (0.0), container.node-5.container_fs_reads./dev/vda1 (0.0), container.node-5.container_fs_writes./dev/vda1 (0.0), container.node-5.container_fs_reads_MB./dev/vda (0.0), container.node-5.container_fs_writes_MB./dev/vda (0.0), container.node-5.container_fs_reads_merged./dev/vda1 (0.0)
- FS metadata / descriptors: container.node-5.container_file_descriptors (0.0 listed among top KPIs earlier)
- Memory metrics: container.node-5.container_memory_cache (17575936.0), container.node-5.container_memory_mapped_file (4096.0), container.node-5.container_memory_max_usage_MB (121.0546875), container.node-5.container_memory_swap (0.0), container.node-5.container_memory_failcnt (0.0)
- Network: container.node-5.container_network_receive_errors.eth0 (0.0), container.node-5.container_network_receive_packets_dropped.eth0 (0.0)

Interpretation / next steps:
- The global P95 and P5 equal the observed values (many zeros or fixed constants), so the anomaly rule flagged all points. This usually indicates a constant series during the whole period — either real constant behavior or an instrumentation/collection issue (e.g., stalled exporter, missing measurements reported as 0, or metric frozen).
- Action items to investigate:
  1. Confirm whether zero/constant values are expected (check application/node health and exporter logs).
  2. Verify metric collection pipeline for frontend-1 / node-5 (scrape success, exporter restart, network issues).
  3. Cross-check traces and error logs for frontend-1 to see if user-visible errors match these telemetry patterns.

Concise conclusion: frontend-1 (node-5 related container metrics) is the primary candidate showing uniform anomalous readings (mostly zeros or fixed constants) across CPU, FS, memory, and network KPIs during the incident window — investigate metric collection and node/container health.

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_cpu_cfs_throttled_periods', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_cpu_cfs_throttled_seconds', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_reads./dev/vda', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_reads_MB./dev/vda', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_writes./dev/vda', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_writes./dev/vda1', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_writes_MB./dev/vda', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_fs_writes_merged./dev/vda1', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_cache', 'global_p95': 17575936.0, 'global_p5': 17575936.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 17575936.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 17575936.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 17575936.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 17575936.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 17575936.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 17575936.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_failcnt', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_failures.container.pgmajfault', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_failures.hierarchy.pgmajfault', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_mapped_file', 'global_p95': 4096.0, 'global_p5': 4096.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 4096.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 4096.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 4096.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 4096.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 4096.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 4096.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_max_usage_MB', 'global_p95': 121.0546875, 'global_p5': 121.0546875, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 121.0546875, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 121.0546875}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 121.0546875}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 121.0546875}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 121.0546875}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 121.0546875}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_memory_swap', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'global_p95': 0.0, 'global_p5': 0.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomaly_values': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}]```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure required columns exist
assert {'timestamp', 'cmdb_id', 'trace_name', 'value'}.issubset(trace_df.columns), "trace.csv missing required columns"

# Target groups: include service-level and pod-level cmdbs for these bases
bases = ['frontend', 'adservice', 'adservice2']  # includes frontend-1/frontend-2 if present as pods
# Incident window (inclusive)
window_start = 1647810000
window_end = 1647811799

# Find candidate cmdb_ids in trace_df that are service-level or pod-level for the bases
all_cmdbs = trace_df['cmdb_id'].astype(str).unique().tolist()
candidates = [cm for cm in all_cmdbs if any(cm == b or cm.startswith(b + '-') for b in bases)]

# timezone for readable timestamps
tz = pytz.timezone('Asia/Shanghai')
def readable(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

rows = []
# compile rules for anomaly type
def anomaly_tail(trace_name):
    name = trace_name.lower()
    # lower-tail indicators
    if ('success_rate' in name) or ('.sr' in name):
        return 'lower'
    # upper-tail indicators
    if ('duration' in name) or ('latency' in name) or ('error' in name) or ('fail' in name) or ('exception' in name):
        return 'upper'
    # default to upper-tail if no explicit rule
    return 'upper'

# Pre-cast types
trace_df['cmdb_id'] = trace_df['cmdb_id'].astype(str)
trace_df['trace_name'] = trace_df['trace_name'].astype(str)
trace_df['value'] = trace_df['value'].astype(float)
trace_df['timestamp'] = trace_df['timestamp'].astype(int)

for cm in candidates:
    sub_cm = trace_df[trace_df['cmdb_id'] == cm]
    if sub_cm.empty:
        continue
    # unique trace_names for this cmdb
    trace_names = sub_cm['trace_name'].unique().tolist()
    for tn in trace_names:
        series = sub_cm[sub_cm['trace_name'] == tn]['value'].dropna().astype(float)
        full_count = int(series.shape[0])
        if full_count == 0:
            continue
        # global thresholds computed on full series before window filtering
        global_P95 = float(np.nanquantile(series.values, 0.95))
        global_P5 = float(np.nanquantile(series.values, 0.05))
        # points in window
        window_df = sub_cm[(sub_cm['trace_name'] == tn) & (sub_cm['timestamp'] >= window_start) & (sub_cm['timestamp'] <= window_end)].copy()
        points_in_window = int(window_df.shape[0])
        tail = anomaly_tail(tn)
        anomaly_count = 0
        earliest_ts = None
        earliest_val = None
        sample_anoms = []
        if points_in_window > 0:
            if tail == 'upper':
                anoms = window_df[window_df['value'] >= global_P95].copy()
            else:
                anoms = window_df[window_df['value'] <= global_P5].copy()
            anomaly_count = int(anoms.shape[0])
            if anomaly_count > 0:
                anoms_sorted_ts = anoms.sort_values('timestamp')
                earliest_row = anoms_sorted_ts.iloc[0]
                earliest_ts = int(earliest_row['timestamp'])
                earliest_val = float(earliest_row['value'])
                # sample up to 5: highest for upper, lowest for lower
                if tail == 'upper':
                    sampled = anoms.sort_values('value', ascending=False).head(5)
                else:
                    sampled = anoms.sort_values('value', ascending=True).head(5)
                sample_anoms = [
                    {"timestamp": int(r['timestamp']),
                     "timestamp_readable": readable(int(r['timestamp'])),
                     "value": float(r['value'])}
                    for _, r in sampled.iterrows()
                ]
        rows.append({
            "cmdb_id": cm,
            "trace_name": tn,
            "full_count": full_count,
            "global_P95": global_P95,
            "global_P5": global_P5,
            "points_in_window": points_in_window,
            "anomaly_count_in_window": anomaly_count,
            "earliest_anomaly_timestamp": int(earliest_ts) if earliest_ts is not None else None,
            "earliest_anomaly_value": float(earliest_val) if earliest_val is not None else None,
            "sample_anomalies": sample_anoms
        })

# Build DataFrame and sort by anomaly_count_in_window desc, then points_in_window desc
result_df = pd.DataFrame(rows)
if result_df.empty:
    result_df = pd.DataFrame(columns=[
        "cmdb_id","trace_name","full_count","global_P95","global_P5",
        "points_in_window","anomaly_count_in_window","earliest_anomaly_timestamp",
        "earliest_anomaly_value","sample_anomalies"
    ])
else:
    result_df = result_df.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False])

# Limit to top 20 rows
summary = result_df.head(20).reset_index(drop=True)

# Add readable earliest timestamp for compactness
summary['earliest_anomaly_timestamp_readable'] = summary['earliest_anomaly_timestamp'].apply(lambda x: readable(x) if pd.notnull(x) else None)

# Return JSON-serializable records (top 20)
result = summary.to_dict(orient='records')

result

```
Out[4]:
```
Summary (plain English):

- What I checked: traces for frontend-1, frontend-2, adservice*, adservice2* (service- and pod-level ids present in the file).

- Main result: Many trace series (mostly *.error_rate traces) across the examined cmdb_ids were flagged as "anomalous" for the entire incident window (1647810000 → 1647811799 / 2022-03-21 05:00–05:29 UTC+8).
  - Examples observed: trace.from_frontend-1.error_rate, trace.from_frontend-2.error_rate, trace.from_root.error_rate, trace.self.error_rate, trace.to_adservice-0.error_rate, trace.to_cartservice-*, trace.to_currencyservice-*, trace.to_productcatalogservice-*, and similar traces on adservice pods (adservice-0, adservice-1, adservice-2, adservice2-0).
  - For each reported component-trace:
    - full_count = 30 (30 samples total)
    - points_in_window = 30 (all samples lie in the incident window)
    - global_P95 and global_P5 = 0.0 (no variability)
    - anomaly_count_in_window = 30 (every sample was flagged)
    - earliest anomaly at 2022-03-21 05:00:00+0800

- Why this happened: the anomaly rules treat error/failure traces as upper-tail (value >= global_P95). Because global_P95 equals 0.0 (the series is constant/zero), the rule marks all points (including zeros) as anomalous. In short, the thresholds are identical to observed values, so the detection flags every point.

- Interpretation and likely cause:
  - This pattern generally indicates constant or frozen trace values (often zeros) rather than true intermittent spikes.
  - Possible causes: instrumentation/tracing exporter problem, missing samples reported as 0, tracing backend ingestion/staleness, or an actual constant zero/error_rate series (the latter is unlikely given the uniformity and anomaly flags).

- Recommended next steps:
  1. Verify trace instrumentation and collection for frontend and adservice pods (check exporter logs, sampling config, and whether traces were actually generated).
  2. Confirm whether zero error_rate is expected; cross-check with application logs and metric telemetry for request counts and error counts.
  3. Inspect tracing pipeline (agent/collector) for restarts, backpressure, or configuration issues that could produce constant values.
  4. Correlate with metrics and error_logs for frontend-1/2 and adservice pods to decide if this is a data-quality problem or a real service behavior issue.

Concise conclusion: traces for frontend and adservice pods show uniform, constant values (global P95/P5 = 0) and therefore every sample in the incident window was flagged. This points to a likely data-collection or instrumentation issue; investigate tracing collection and correlate with logs/metrics.

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice-1', 'trace_name': 'trace.from_frontend-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice-2', 'trace_name': 'trace.from_frontend-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'adservice2-0', 'trace_name': 'trace.from_frontend2-0.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.from_root.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.self.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_adservice-0.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_adservice-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_adservice-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_cartservice-0.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_cartservice-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_cartservice-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_currencyservice-0.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_currencyservice-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_currencyservice-2.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_productcatalogservice-0.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}, {'cmdb_id': 'frontend-1', 'trace_name': 'trace.to_productcatalogservice-1.error_rate', 'full_count': 30, 'global_P95': 0.0, 'global_P5': 0.0, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 0.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 0.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 0.0}], 'earliest_anomaly_timestamp_readable': '2022-03-21 05:00:00+0800'}]```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing DataFrames if present
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure types
error_df['timestamp'] = error_df['timestamp'].astype(int)
error_df['cmdb_id'] = error_df['cmdb_id'].astype(str)
if 'message' in error_df.columns:
    error_df['message'] = error_df['message'].astype(str)
else:
    error_df['message'] = error_df.get('message', pd.Series(dtype=str))

log_df['timestamp'] = log_df['timestamp'].astype(int)
log_df['cmdb_id'] = log_df['cmdb_id'].astype(str)
log_df['log_name'] = log_df['log_name'].astype(str)
# ensure 'value' exists and is string-safe for searching
log_df['value'] = log_df['value'].astype(str)

# timezone
tz = pytz.timezone('Asia/Shanghai')
def to_readable(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

# 1) error_logs.csv summary
err_total_rows = int(len(error_df))
err_distinct_cmdb_count = int(error_df['cmdb_id'].nunique())
err_cmdb_top = (
    error_df['cmdb_id']
    .value_counts()
    .head(20)
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# messages: top 10 by frequency, with first occurrence info
msg_counts = error_df['message'].value_counts().head(10).rename_axis('message').reset_index(name='count')
msgs_info = []
for _, r in msg_counts.iterrows():
    msg = r['message']
    subset = error_df[error_df['message'] == msg]
    first_row = subset.sort_values('timestamp').iloc[0]
    first_ts = int(first_row['timestamp'])
    msgs_info.append({
        "message": msg,
        "count": int(r['count']),
        "first_timestamp": first_ts,
        "first_timestamp_readable": to_readable(first_ts),
        "first_cmdb_id": first_row['cmdb_id']
    })

# earliest overall error log timestamp
if err_total_rows > 0:
    earliest_err_ts = int(error_df['timestamp'].min())
    earliest_err_ts_readable = to_readable(earliest_err_ts)
else:
    earliest_err_ts = None
    earliest_err_ts_readable = None

error_summary = {
    "total_rows": err_total_rows,
    "distinct_cmdb_count": err_distinct_cmdb_count,
    "top_20_cmdb_counts": err_cmdb_top.head(20).to_dict(orient='records'),
    "top_up_to_10_messages_with_first_observed": msgs_info,
    "earliest_error_timestamp": earliest_err_ts,
    "earliest_error_timestamp_readable": earliest_err_ts_readable
}

# 2) log.csv filtered summary
filter_cmdbs = ['frontend-1','frontend-2','adservice','adservice2','adservice-0','adservice-1','adservice-2','adservice2-0']
filtered_logs = log_df[log_df['cmdb_id'].isin(filter_cmdbs)].copy()

# per-cmdb counts
per_cmdb_counts = (
    filtered_logs['cmdb_id']
    .value_counts()
    .reindex(filter_cmdbs, fill_value=0)  # keep order consistent
    .reset_index()
    .rename(columns={'index':'cmdb_id','cmdb_id':'count'})
)

# top log_name per cmdb (top 10)
top_logs_by_cmdb = {}
for cm in filter_cmdbs:
    sub = filtered_logs[filtered_logs['cmdb_id'] == cm]
    if sub.empty:
        top_logs_by_cmdb[cm] = []
        continue
    top10 = sub['log_name'].value_counts().head(10).rename_axis('log_name').reset_index(name='count')
    top_logs_by_cmdb[cm] = top10.to_dict(orient='records')

# search for rows where log_name or value contains keywords
import re
pattern = re.compile(r'error|fail|exception|ads', flags=re.IGNORECASE)
mask = filtered_logs['log_name'].str.contains(pattern) | filtered_logs['value'].str.contains(pattern)
matched = filtered_logs[mask].copy().sort_values('timestamp').head(20)
# add readable timestamp
if not matched.empty:
    matched['timestamp_readable'] = matched['timestamp'].apply(to_readable)
    matched_rows = matched[['timestamp','timestamp_readable','cmdb_id','log_name','value']].to_dict(orient='records')
else:
    matched_rows = []

log_summary = {
    "filtered_cmdbs": filter_cmdbs,
    "per_cmdb_counts": per_cmdb_counts.to_dict(orient='records'),
    "top_log_name_per_cmdb_top10": top_logs_by_cmdb,
    "matched_log_rows_up_to_20": matched_rows
}

summary = {
    "error_logs.csv": error_summary,
    "log.csv_filtered_summary": log_summary
}

summary

```
Out[5]:
```
Summary (plain English):

- error_logs.csv
  - Total error log rows: 453.
  - Distinct cmdb_ids: 2 — frontend-1 (232 rows) and frontend-2 (221 rows).
  - Single dominant message (453 occurrences): "severity: warning, message: failed to retrieve ads".
    - First observed at 2022-03-21 05:00:04 (UTC+8) on frontend-1.
  - Earliest error log timestamp in the file: 2022-03-21 05:00:04 (UTC+8).

- log.csv (filtered to frontends and adservice pods)
  - Filtered cmdbs and row counts:
    - frontend-1: 60 rows
    - frontend-2: 60 rows
    - adservice: 0 rows
    - adservice2: 0 rows
    - adservice-0: 60 rows
    - adservice-1: 60 rows
    - adservice-2: 60 rows
    - adservice2-0: 0 rows
  - Top log_names per populated cmdb (top 10 shown):
    - frontend-1: log.error_count (30), log.row_count (30)
    - frontend-2: log.error_count (30), log.row_count (30)
    - adservice-0 / adservice-1 / adservice-2: log.error_count (30), log.row_count (30)
  - Rows matching keywords (error|fail|exception|ads), up to 20 samples:
    - Frontend entries show non-zero log.error_count values (examples at 05:00:00 UTC+8: frontend-1 = 9.0, frontend-2 = 9.0; later times show 7.0, 8.0, 10.0, etc.).
    - adservice-* pods in these matched samples show log.error_count = 0.0 at the same timestamps.

- Interpretation / concise takeaways
  - All error log messages point to frontends failing to retrieve ads; the error is observed first on frontend-1 at 05:00:04 and appears across both frontends.
  - Frontends show elevated log.error_count values during the window, while adservice pods sampled here report zero log errors.
  - This pattern suggests the failing symptom is reported at the frontend layer ("failed to retrieve ads") rather than clear errors inside adservice pods in the sampled log metrics.
  - Recommended next steps: check frontend stack traces/metrics and networking to adservice, verify adservice health (service-level traces/metrics beyond these log.error_count rows), and inspect any network timeouts or upstream failures between frontends and adservice.

The original code execution output of IPython Kernel is also provided below for reference:

{'error_logs.csv': {'total_rows': 453, 'distinct_cmdb_count': 2, 'top_20_cmdb_counts': [{'cmdb_id': 'frontend-1', 'count': 232}, {'cmdb_id': 'frontend-2', 'count': 221}], 'top_up_to_10_messages_with_first_observed': [{'message': 'severity: warning, message: failed to retrieve ads', 'count': 453, 'first_timestamp': 1647810004, 'first_timestamp_readable': '2022-03-21 05:00:04+0800', 'first_cmdb_id': 'frontend-1'}], 'earliest_error_timestamp': 1647810004, 'earliest_error_timestamp_readable': '2022-03-21 05:00:04+0800'}, 'log.csv_filtered_summary': {'filtered_cmdbs': ['frontend-1', 'frontend-2', 'adservice', 'adservice2', 'adservice-0', 'adservice-1', 'adservice-2', 'adservice2-0'], 'per_cmdb_counts': [{'count': 60}, {'count': 60}, {'count': 0}, {'count': 0}, {'count': 60}, {'count': 60}, {'count': 60}, {'count': 0}], 'top_log_name_per_cmdb_top10': {'frontend-1': [{'log_name': 'log.error_count', 'count': 30}, {'log_name': 'log.row_count', 'count': 30}], 'frontend-2': [{'log_name': 'log.error_count', 'count': 30}, {'log_name': 'log.row_count', 'count': 30}], 'adservice': [], 'adservice2': [], 'adservice-0': [{'log_name': 'log.error_count', 'count': 30}, {'log_name': 'log.row_count', 'count': 30}], 'adservice-1': [{'log_name': 'log.error_count', 'count': 30}, {'log_name': 'log.row_count', 'count': 30}], 'adservice-2': [{'log_name': 'log.error_count', 'count': 30}, {'log_name': 'log.row_count', 'count': 30}], 'adservice2-0': []}, 'matched_log_rows_up_to_20': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'cmdb_id': 'frontend-1', 'log_name': 'log.error_count', 'value': '9.0'}, {'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'cmdb_id': 'frontend-2', 'log_name': 'log.error_count', 'value': '9.0'}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'cmdb_id': 'frontend-1', 'log_name': 'log.error_count', 'value': '7.0'}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'cmdb_id': 'frontend-2', 'log_name': 'log.error_count', 'value': '4.0'}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'cmdb_id': 'frontend-1', 'log_name': 'log.error_count', 'value': '8.0'}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'cmdb_id': 'frontend-2', 'log_name': 'log.error_count', 'value': '10.0'}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': '0.0'}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'cmdb_id': 'frontend-1', 'log_name': 'log.error_count', 'value': '7.0'}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'cmdb_id': 'frontend-2', 'log_name': 'log.error_count', 'value': '6.0'}]}}```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz

# reuse metric_df loaded earlier
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure columns types
metric_df['cmdb_id'] = metric_df['cmdb_id'].astype(str)
metric_df['kpi_name'] = metric_df['kpi_name'].astype(str)
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = metric_df['value'].astype(float)

# Parameters
candidates = ['frontend-1','frontend-2','frontend','adservice','adservice-0','adservice-1','adservice-2']
kpis_of_interest = [
    'app.http.count','app.grpc.count',
    'app.http.sr','app.grpc.sr',
    'app.http.mrt','app.grpc.mrt'
]
window_start = 1647810000
window_end = 1647811799

tz = pytz.timezone('Asia/Shanghai')
def readable(ts):
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

rows = []
# Iterate components and KPIs, compute required stats
for cm in candidates:
    for kp in kpis_of_interest:
        sub = metric_df[(metric_df['cmdb_id'] == cm) & (metric_df['kpi_name'] == kp)]
        if sub.empty:
            continue
        series = sub['value'].dropna().astype(float)
        full_count = int(series.shape[0])
        if full_count == 0:
            continue
        # global thresholds on full series (rule 9)
        global_P95 = float(np.nanquantile(series.values, 0.95))
        global_P5 = float(np.nanquantile(series.values, 0.05))
        # points in incident window
        window_df = sub[(sub['timestamp'] >= window_start) & (sub['timestamp'] <= window_end)].copy()
        points_in_window = int(window_df.shape[0])
        # Determine anomaly tail per KPI rules
        name_lower = kp.lower()
        if 'mrt' in name_lower:
            tail = 'upper'   # latency spike
        elif '.sr' in name_lower or 'sr' in name_lower:
            tail = 'lower'   # success rate drop
        elif 'count' in name_lower:
            tail = 'lower'   # traffic drop
        else:
            tail = 'upper'   # default
        anomaly_count = 0
        earliest_ts = None
        earliest_val = None
        sample_anoms = []
        if points_in_window > 0:
            if tail == 'upper':
                anoms = window_df[window_df['value'] >= global_P95].copy()
            else:
                anoms = window_df[window_df['value'] <= global_P5].copy()
            anomaly_count = int(anoms.shape[0])
            if anomaly_count > 0:
                anoms_sorted_ts = anoms.sort_values('timestamp')
                earliest_row = anoms_sorted_ts.iloc[0]
                earliest_ts = int(earliest_row['timestamp'])
                earliest_val = float(earliest_row['value'])
                # sample up to 5 points: highest for upper, lowest for lower
                if tail == 'upper':
                    sampled = anoms.sort_values('value', ascending=False).head(5)
                else:
                    sampled = anoms.sort_values('value', ascending=True).head(5)
                sample_anoms = [
                    {"timestamp": int(r['timestamp']), "timestamp_readable": readable(int(r['timestamp'])), "value": float(r['value'])}
                    for _, r in sampled.iterrows()
                ]
        rows.append({
            "cmdb_id": cm,
            "kpi_name": kp,
            "global_P95": global_P95,
            "global_P5": global_P5,
            "full_count": full_count,
            "points_in_window": points_in_window,
            "anomaly_count_in_window": anomaly_count,
            "earliest_anomaly_timestamp": int(earliest_ts) if earliest_ts is not None else None,
            "earliest_anomaly_value": float(earliest_val) if earliest_val is not None else None,
            "sample_anomalies": sample_anoms
        })

# Build DataFrame, sort by anomaly_count desc then points_in_window, limit top 20
df = pd.DataFrame(rows)
if df.empty:
    result = []
else:
    df_sorted = df.sort_values(['anomaly_count_in_window','points_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
    result = df_sorted.to_dict(orient='records')

result

```
Out[6]:
```
Summary (plain English):

What I checked
- Components: frontend-1, frontend-2, frontend (service-level), adservice, adservice-0, adservice-1, adservice-2.
- KPIs looked for: app.http.count, app.grpc.count, app.http.sr, app.grpc.sr, app.http.mrt, app.grpc.mrt.

Which component-KPIs exist (and key findings)
- frontend (service-level)
  - app.http.sr — full_count=30, global_P95=global_P5=100.0; points_in_window=30; anomaly_count_in_window=30 (every point flagged because global P5==P95==100 and anomaly rule is value <= P5). Earliest anomaly at 2022-03-21 05:00:00+08. Sample anomalies include values ~100.0 and one 99.21.
  - app.http.count — full_count=30; global_P95≈135.55, global_P5≈115.45; points_in_window=30; anomaly_count_in_window=2 (two low-traffic points below P5). Earliest traffic-drop anomaly at 2022-03-21 05:12:00+08 (value 115.0). Sample low points: 115.0 and 112.0.
  - app.http.mrt — full_count=30; global_P95≈0.2118, global_P5≈0.04918; points_in_window=30; anomaly_count_in_window=2 (two latency spikes >= P95). Earliest latency anomaly at 2022-03-21 05:11:00+08 (value ≈0.2123). Sample spikes ~0.212–0.224 s.

- adservice (service-level)
  - app.http.count — full_count=30; global_P95=global_P5=24.0; points_in_window=30; anomaly_count_in_window=30 (all points flagged because P5==P95==24.0). Earliest anomaly at 05:00:00+08 (value 24.0).
  - app.http.sr — full_count=30; global_P95=global_P5=100.0; points_in_window=30; anomaly_count_in_window=30 (all points flagged). Earliest anomaly at 05:00:00+08 (value 100.0).
  - app.grpc.count — full_count=30; global_P95=85.0, global_P5≈66.9; points_in_window=30; anomaly_count_in_window=2 (two low-traffic points). Earliest low at 05:02:00+08 (value 64.0). Sample lows: 64.0 and 66.0.
  - app.grpc.sr — full_count=30; global_P95≈83.02, global_P5≈77.60; points_in_window=30; anomaly_count_in_window=2 (two low success-rate points). Earliest at 05:17:00+08 (value ≈77.33). Sample lows ≈77.03 and 77.33.
  - app.http.mrt — full_count=30; global_P95≈0.009658, global_P5≈0.009108; points_in_window=30; anomaly_count_in_window=2 (latency spikes). Earliest at 05:16:00+08.
  - app.grpc.mrt — full_count=30; global_P95≈0.002685, global_P5≈0.002390; points_in_window=30; anomaly_count_in_window=2 (latency spikes). Earliest at 05:00:00+08.

Notes on interpretation
- Several KPIs have global_P95 == global_P5 (constant series). Where that happens (e.g., app.http.sr = 100.0 or app.http.count = 24.0), the anomaly rule (<= P5 for counts/sr) causes every sample to be flagged. That typically indicates a constant/frozen series or limited variability rather than clear degradations.
- For the KPIs with small anomaly counts (mostly 2), these represent real deviations during the incident window:
  - frontend app.http.count drop at ~05:12 and ~05:16.
  - frontend app.http.mrt spikes ~05:11–05:14.
  - adservice app.grpc.count and app.grpc.sr show low points (e.g., 05:02 and 05:17).
  - adservice latency KPIs (http.grpc.mrt, grpc.mrt) show spikes at specific times.

Practical next steps
1. Treat the "all points flagged" KPIs as potential data-quality/constant-series signals — verify instrumentation and whether constant values are expected.
2. For the KPIs with a small number of anomalies (traffic drops and latency spikes), investigate the specific timestamps (approx. 05:00–05:24 window, with notable events at ~05:02, ~05:11–05:17, ~05:12–05:16).
3. Correlate these timestamps with traces and error logs (frontends reported "failed to retrieve ads" starting ~05:00:04) to see causal relationships between frontend errors, traffic drops, and adservice success/latency metrics.

Concise conclusion
- There are both data-quality-like constant series (leading to all-sample flags) and a few concrete anomalies (traffic drops and latency spikes) in the incident window. Focus investigation on the specific timestamps noted and verify both metric collection and service behavior (frontends ↔ adservice).

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'frontend', 'kpi_name': 'app.http.sr', 'global_P95': 100.0, 'global_P5': 100.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 100.0, 'sample_anomalies': [{'timestamp': 1647810480, 'timestamp_readable': '2022-03-21 05:08:00+0800', 'value': 99.21259842519686}, {'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 100.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 100.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 100.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 100.0}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'global_P95': 24.0, 'global_P5': 24.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 24.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 24.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 24.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 24.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 24.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 24.0}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.sr', 'global_P95': 100.0, 'global_P5': 100.0, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 30, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 100.0, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 100.0}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 100.0}, {'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 100.0}, {'timestamp': 1647810180, 'timestamp_readable': '2022-03-21 05:03:00+0800', 'value': 100.0}, {'timestamp': 1647810240, 'timestamp_readable': '2022-03-21 05:04:00+0800', 'value': 100.0}]}, {'cmdb_id': 'frontend', 'kpi_name': 'app.http.count', 'global_P95': 135.55, 'global_P5': 115.45, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647810720, 'earliest_anomaly_value': 115.0, 'sample_anomalies': [{'timestamp': 1647810960, 'timestamp_readable': '2022-03-21 05:16:00+0800', 'value': 112.0}, {'timestamp': 1647810720, 'timestamp_readable': '2022-03-21 05:12:00+0800', 'value': 115.0}]}, {'cmdb_id': 'frontend', 'kpi_name': 'app.http.mrt', 'global_P95': 0.21180849248798056, 'global_P5': 0.04918429225352143, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647810660, 'earliest_anomaly_value': 0.2122582031249999, 'sample_anomalies': [{'timestamp': 1647810840, 'timestamp_readable': '2022-03-21 05:14:00+0800', 'value': 0.2244183593749999}, {'timestamp': 1647810660, 'timestamp_readable': '2022-03-21 05:11:00+0800', 'value': 0.2122582031249999}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'global_P95': 85.0, 'global_P5': 66.9, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647810120, 'earliest_anomaly_value': 64.0, 'sample_anomalies': [{'timestamp': 1647810120, 'timestamp_readable': '2022-03-21 05:02:00+0800', 'value': 64.0}, {'timestamp': 1647810720, 'timestamp_readable': '2022-03-21 05:12:00+0800', 'value': 66.0}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'global_P95': 83.01691729323308, 'global_P5': 77.5982683982684, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647811020, 'earliest_anomaly_value': 77.33333333333333, 'sample_anomalies': [{'timestamp': 1647811200, 'timestamp_readable': '2022-03-21 05:20:00+0800', 'value': 77.02702702702703}, {'timestamp': 1647811020, 'timestamp_readable': '2022-03-21 05:17:00+0800', 'value': 77.33333333333333}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.http.mrt', 'global_P95': 0.009658333333338615, 'global_P5': 0.009108229166664605, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647810960, 'earliest_anomaly_value': 0.0096958333333386, 'sample_anomalies': [{'timestamp': 1647811440, 'timestamp_readable': '2022-03-21 05:24:00+0800', 'value': 0.00979791666667}, {'timestamp': 1647810960, 'timestamp_readable': '2022-03-21 05:16:00+0800', 'value': 0.0096958333333386}]}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'global_P95': 0.0026846070460701398, 'global_P5': 0.00239035252644093, 'full_count': 30, 'points_in_window': 30, 'anomaly_count_in_window': 2, 'earliest_anomaly_timestamp': 1647810000, 'earliest_anomaly_value': 0.0026987804878033, 'sample_anomalies': [{'timestamp': 1647810000, 'timestamp_readable': '2022-03-21 05:00:00+0800', 'value': 0.0026987804878033}, {'timestamp': 1647810060, 'timestamp_readable': '2022-03-21 05:01:00+0800', 'value': 0.0026987804878033}]}]```
```